# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR^2 colorectal cancer dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema and accessible via:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print overview
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print("Keywords:", ', '.join(metadata.keywords))

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR^2 colorectal cancer dataset follows the Croissant schema. Each record set, field, and column is uniquely identified by its `@id`.

Let's enumerate the dataset's record sets and their fields.

In [ ]:
# Find available record sets and their @ids
record_sets = dataset.record_sets
print("Available record sets:")
for rset in record_sets:
    print(f"  - Name: {rset.name}, @id: {rset.id}")

# For demonstration, print fields for first record set
if record_sets:
    main_record_set = record_sets[0]
    print("\nFields in selected record set:")
    for field in main_record_set.fields:
        print(f"  - Field Name: {field.name}, @id: {field.id}, Data Type: {field.data_type}")

## 2.1 Records Example from a Record Set
Let's preview a few records by referencing one record set using its `@id`.

In [ ]:
# Preview records from the main record set (by @id)
ds = dataset
main_record_set_id = main_record_set.id

# Print a few sample records
for i, record in enumerate(ds.records(record_set=main_record_set_id)):
    print(f"Record {i+1}: {record}")
    if i >= 2:
        break

## 3. Data Extraction
Load data from all record sets into pandas DataFrames for analysis.

All entities are referenced by their `@id` fields.

In [ ]:
# Extract data from each record set
record_set_ids = [rset.id for rset in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Show columns in first record set
print(f"Columns for record set '@id': {main_record_set_id}")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps with pandas.

- Filter records based on a numeric field (`@id`)
- Normalize numeric values
- Group by categorical field (`@id`)


In [ ]:
# Select a numeric and a group field
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Identify numeric and group fields for demonstration
numeric_fields = [field.id for field in main_record_set.fields if field.data_type in ['Integer', 'Float', 'Number']]
group_fields = [field.id for field in main_record_set.fields if field.data_type == 'Text' or field.data_type == 'Boolean']

# Use the first numeric and group field found
if numeric_fields:
    numeric_field_id = numeric_fields[0]
else:
    numeric_field_id = df.select_dtypes('number').columns[0]

if group_fields:
    group_field_id = group_fields[0]
else:
    group_field_id = df.select_dtypes('object').columns[0]

threshold = 50
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group
    if group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped statistics by '{group_field_id}':")
        print(grouped.head())

## 5. Visualization
Visualize distributions or relationships between fields.

Below, we visualize the distribution of a numeric field and its relation to a group field. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

if group_field_id in df.columns and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} distribution grouped by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 colorectal cancer dataset using `mlcroissant`, referenced all entities by their `@id`, and performed basic EDA and visualization.

- The dataset includes multiple clinicopathological variables for cancer survivors with second primary colorectal cancer.
- Tabular data was extracted by record set and analyzed for numeric and categorical features.
- `mlcroissant` enabled seamless metadata and data extraction directly from the Croissant schema.

Further analysis can be performed to investigate biomarker distributions, anatomical patterns, and outcome predictors in this cohort.